# viz_important — Top 10 Charts

**Version:** `v2_algorithmic`  
**Output:** `pres/viz/`

Reads from version's own `intermediary/` if populated; falls back to shared root `intermediary/` otherwise.

## 0. Setup

In [1]:
import os
ROOT      = "/Users/leoss/Desktop/GitHub/Capstone/FINAL CODE RECAP/v2_algorithmic"
SHARED    = "/Users/leoss/Desktop/GitHub/Capstone/FINAL CODE RECAP"
_marker   = os.path.join(ROOT, "intermediary", "Master.csv")
DATA_ROOT = ROOT if os.path.exists(_marker) else SHARED
OUT       = os.path.join(ROOT, "pres", "viz")
os.makedirs(OUT, exist_ok=True)
print(f"ROOT:      {ROOT}")
print(f"DATA_ROOT: {DATA_ROOT}")
print(f"OUT:       {OUT}")


ROOT:      /Users/leoss/Desktop/GitHub/Capstone/FINAL CODE RECAP/v2_algorithmic
DATA_ROOT: /Users/leoss/Desktop/GitHub/Capstone/FINAL CODE RECAP/v2_algorithmic
OUT:       /Users/leoss/Desktop/GitHub/Capstone/FINAL CODE RECAP/v2_algorithmic/pres/viz


In [2]:
import os, sys, math, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.decomposition import PCA

# ── Project root ─────────────────────────────────────────────────────────────

# ── Output directory ──────────────────────────────────────────────────────────

# ── Style constants ───────────────────────────────────────────────────────────
FONT = 'IBM Plex Sans, -apple-system, BlinkMacSystemFont, sans-serif'
BG   = '#ffffff'
NAVY = '#1a2744'
GRID = '#e5e7eb'
WRITE_CONFIG = {'displayModeBar': False, 'responsive': True}

# ── Cluster colors (k=4, matching NB4 label assignment) ──────────────────────
CLUSTER_COLORS = {
    'Petrostates':        '#E63946',
    'Oil Exporters':      '#457B9D',
    'Major Producers':    '#2A9D8F',
    'Mining Exporters':   '#E9C46A',
    'Forestry Intensive': '#8B5CF6',
}

# ── Helper functions ──────────────────────────────────────────────────────────
def base_layout(**kw):
    d = dict(template='plotly_white', plot_bgcolor=BG, paper_bgcolor=BG,
             font=dict(family=FONT, size=11, color=NAVY),
             margin=dict(l=60, r=40, t=40, b=50))
    d.update(kw)
    return d

def save(fig, name, w=1100, h=600):
    path = os.path.join(OUT, name)
    try:
        fig.write_image(f"{path}.png", width=w, height=h, scale=2)
        print(f"  Saved: {path}.png")
    except Exception as e:
        print(f"  PNG skipped: {e}")

_FEAT_SHORT = {
    'Domestic credit to private sector (% of GDP)':                        'Domestic Credit',
    'Access to electricity (% of population)':                             'Electricity Access',
    'Human capital index':                                                  'Human Capital',
    'HCI_x_ProductionValue':                                               'HC x Production',
    'GFCF_x_ProductionValue':                                              'GFCF x Production',
    'Rule of law index':                                                    'Rule of Law',
    'Political stability \u2014 estimate':                                  'Political Stability',
    'Trade (% of GDP)':                                                     'Trade',
    'Gross fixed capital formation, all, Constant prices, Percent of GDP': 'Capital Formation',
    'Urban population (% of total population)':                           'Urban Population',
    'Use of IMF credit (DOD, current US$)':                               'IMF Credit',
    'Total_Production_Value_Per_Capita':                                   'Prod Value p.c.',
    'Capital depreciation rate':                                           'Depreciation',
    'Landlocked':                                                          'Landlocked',
    'Real interest rate (%)':                                              'Interest Rate',
    'Inflation, consumer prices (annual %)':                               'Inflation',
    'GDP per capita (constant prices, PPP)':                               'GDP per Capita',
    'Total natural resources rents (% of GDP)':                           'NR Rents (% GDP)',
    'Oil rents (% of GDP)':                                                'Oil Rents',
    'Mineral rents (% of GDP)':                                            'Mineral Rents',
    'Natural gas rents (% of GDP)':                                        'Gas Rents',
    'Economic Complexity Index':                                           'ECI',
    'Adjusted savings: gross savings (% of GNI)':                        'Savings',
    'Government revenue':                                                   'Gov Revenue',
    'Share of investment in GDP':                                          'Investment Share',
    'Life expectancy at birth, total (years)':                            'Life Expectancy',
    'Death rate, crude (per 1,000 people)':                               'Death Rate',
    'Manufacturing, value added (% of GDP)':                              'Manufacturing',
    'Agriculture, forestry, and fishing, value added (% of GDP)':        'Agriculture (% GDP)',
    'Mobile cellular subscriptions (per 100 people)':                     'Mobile Subs',
    'Political corruption index':                                          'Pol. Corruption',
    'Property rights':                                                     'Property Rights',
    'Services, value added (% of GDP)':                                   'Services (% GDP)',
    'Industry (including construction), value added (% of GDP)':         'Industry (% GDP)',
    'prod_pc':                                                             'Prod Value p.c.',
}
def shorten(name, max_len=30):
    return _FEAT_SHORT.get(name, name[:max_len])

# ── Load data ─────────────────────────────────────────────────────────────────
master    = pd.read_csv(os.path.join(DATA_ROOT, 'intermediary/Master.csv'))
nr_full   = pd.read_csv(os.path.join(DATA_ROOT, 'intermediary/NaturalResource.csv'))
include   = pd.read_csv(os.path.join(DATA_ROOT, 'intermediary/sample_countries_final.csv'))['Country Code'].tolist()
clusters  = pd.read_csv(os.path.join(DATA_ROOT, 'intermediary/clustersagg.csv'))

panel = master[(master['Country Code'].isin(include)) &
               (master['Year'] >= 1995) & (master['Year'] <= 2019)].copy()
nr_sample = nr_full[nr_full['Country Code'].isin(include)]

# Attach cluster labels to panel
cl_map = clusters[['Country Code', 'Country', 'Cluster', 'ClusterLabels']].drop_duplicates('Country Code')
panel = panel.merge(cl_map, on='Country Code', how='left')

print(f"Panel: {panel['Country Code'].nunique()} countries, {len(panel):,} obs")
print(f"NR data: {nr_sample['Country Code'].nunique()} countries, {len(nr_sample):,} rows")
print(f"Clusters: {clusters['ClusterLabels'].value_counts().to_dict()}")

Panel: 75 countries, 1,875 obs
NR data: 75 countries, 9,517 rows
Clusters: {'Forestry Intensive': 37, 'Oil Exporters': 16, 'Major Producers': 10, 'Petrostates': 7, 'Mining Exporters': 4}


In [3]:
def save(fig, name, w=1100, h=600):
    path = os.path.join(OUT, name)
    fig.write_image(f"{path}.png", width=w, height=h, scale=2)
    fig.write_html(f"{path}.html", config=WRITE_CONFIG)
    print(f"Saved: {path}.png")


### Chart 1 — Sample Map

In [4]:
# Chart 000 — World map highlighting all countries in the sample
SAMPLE_COLOR = '#457B9D'

# Build a simple 0/1 dataframe: 1 = in sample
df_sample = (
    master[['Country Code', 'Country Name']].drop_duplicates('Country Code')
    .query('`Country Code` in @include')
    .assign(z=1)
)

fig000 = go.Figure()

fig000.add_trace(go.Choropleth(
    locations=df_sample['Country Code'],
    z=df_sample['z'],
    colorscale=[[0, SAMPLE_COLOR], [1, SAMPLE_COLOR]],
    showscale=False,
    showlegend=False,
    hovertemplate='<b>%{text}</b><extra></extra>',
    text=df_sample['Country Name'].values,
    marker=dict(line=dict(color='white', width=0.6)),
))

fig000.update_layout(
    **base_layout(
        title=dict(
            text='Sample countries (n={})'.format(len(df_sample)),
            x=0.5, xanchor='center',
            font=dict(size=15, color=NAVY),
        ),
        geo=dict(
            showframe=False,
            showcoastlines=False,
            showland=True,  landcolor='#f4f4f4',
            showocean=True, oceancolor='#ffffff',
            showlakes=False,
            bgcolor=BG,
            projection_type='natural earth',
        ),
        margin=dict(l=0, r=0, t=50, b=0),
        height=460,
    )
)

fig000.show(config=WRITE_CONFIG)
save(fig000, '01_sample_map', w=1200, h=500)

Saved: /Users/leoss/Desktop/GitHub/Capstone/FINAL CODE RECAP/v2_algorithmic/pres/viz/01_sample_map.png


### Chart 2 — PCA Biplot

In [5]:
# PCA on 1995 resource production data (per-capita, log-scaled)
nr_1995 = nr_sample[nr_sample['Year'] == 1995]

pivot = nr_1995.pivot_table(
    index=['Country', 'Country Code', 'Year', 'Population'],
    columns='Resource', values='Production_TotalValue',
).reset_index()

res_cols = [c for c in pivot.columns
            if c not in ['Country', 'Country Code', 'Year', 'Population']]
pivot[res_cols] = pivot[res_cols].div(pivot['Population'], axis=0)
pivot = pivot.fillna(0)

X = np.log1p(pivot[res_cols].fillna(0))
pca = PCA(n_components=2, random_state=42)
pca.fit(X)

var1 = pca.explained_variance_ratio_[0] * 100
var2 = pca.explained_variance_ratio_[1] * 100
print(f"PCA: var explained = {var1:.1f}% / {var2:.1f}%")


PCA: var explained = 49.1% / 16.7%


In [6]:
cl_1995 = pd.read_csv(os.path.join(DATA_ROOT, 'intermediary/clusters1995.csv'))

# Recompute PCA for biplot arrows (need loadings)
loadings_biplot = pd.DataFrame(
    pca.components_.T * np.sqrt(pca.explained_variance_),
    columns=['PC1', 'PC2'], index=res_cols,
)
importance = loadings_biplot.abs().sum(axis=1)
top5  = importance.nlargest(5).index
top10 = importance.nlargest(10).index
scale = 2.8

fig03 = go.Figure()

for lbl in sorted(cl_1995['ClusterLabels'].unique()):
    sub = cl_1995[cl_1995['ClusterLabels'] == lbl]
    color = CLUSTER_COLORS.get(lbl, '#999')
    fig03.add_trace(go.Scatter(
        x=sub['PC1'], y=sub['PC2'], mode='markers+text',
        marker=dict(size=10, color=color, opacity=0.82,
                    line=dict(width=1.2, color='white')),
        text=sub['Country Code'], textposition='top center',
        textfont=dict(size=8, color='#333'), name=lbl,
        hovertemplate='<b>%{text}</b><br>PC1=%{x:.2f}, PC2=%{y:.2f}<extra></extra>',
    ))

# Biplot arrows
for feat in top10:
    is_top5 = feat in top5
    x1 = loadings_biplot.loc[feat, 'PC1'] * scale
    y1 = loadings_biplot.loc[feat, 'PC2'] * scale
    fig03.add_annotation(
        x=x1, y=y1, ax=0, ay=0,
        xref='x', yref='y', axref='x', ayref='y',
        showarrow=True,
        arrowhead=3 if is_top5 else 2,
        arrowsize=1.2 if is_top5 else 0.8,
        arrowwidth=2.2 if is_top5 else 1.2,
        arrowcolor='#222' if is_top5 else 'rgba(150,150,150,0.5)',
    )
    if is_top5:
        fig03.add_annotation(
            x=x1 * 1.18, y=y1 * 1.18,
            text=f'<b>{feat}</b>', showarrow=False,
            font=dict(size=10, color='#111', family=FONT),
            bgcolor='rgba(255,255,255,0.7)', borderpad=2,
        )

fig03.add_hline(y=0, line=dict(color=GRID, width=1))
fig03.add_vline(x=0, line=dict(color=GRID, width=1))

fig03.update_layout(**base_layout(
    height=680, margin=dict(l=60, r=60, t=50, b=60),
    xaxis=dict(title=f'PC1 ({var1:.1f}% variance)', gridcolor=GRID, gridwidth=0.5),
    yaxis=dict(title=f'PC2 ({var2:.1f}% variance)', gridcolor=GRID, gridwidth=0.5),
    legend=dict(title='Resource profile (k=4, 1995)', font=dict(size=10),
                bgcolor='rgba(250,250,250,0.85)', bordercolor=GRID, borderwidth=1),
))
save(fig03, '02_pca_biplot', w=1100, h=680)

Saved: /Users/leoss/Desktop/GitHub/Capstone/FINAL CODE RECAP/v2_algorithmic/pres/viz/02_pca_biplot.png


### Chart 3 — Cluster World Map

In [7]:
def create_cluster_map(cl_df):
    fig = go.Figure()
    for lbl in sorted(cl_df['ClusterLabels'].unique()):
        sub   = cl_df[cl_df['ClusterLabels'] == lbl]
        color = CLUSTER_COLORS.get(lbl, '#aaa')
        fig.add_trace(go.Choropleth(
            locations=sub['Country Code'],
            z=[1] * len(sub),
            colorscale=[[0, color], [1, color]],
            showscale=False,
            showlegend=True,
            name=lbl,
            text=sub['Country'],
            hovertemplate='<b>%{text}</b><br>' + lbl + '<extra></extra>',
            marker=dict(line=dict(color='white', width=0.6)),
        ))

    fig.update_geos(
        projection_type='natural earth',
        showcountries=True, countrycolor='#d0d0d0',
        showcoastlines=False,
        showland=True, landcolor='#f0f0f0',
        showocean=True, oceancolor='#dde8f0',
        showframe=False,
    )
    fig.update_layout(
        margin=dict(l=0, r=0, t=30, b=60),
        legend=dict(
            orientation='h', x=0.5, y=-0.06,
            xanchor='center', yanchor='top',
            font=dict(size=11, family=FONT),
            bgcolor='rgba(250,250,250,0.9)',
            bordercolor='#d0d0d0', borderwidth=1,
        ),
        paper_bgcolor=BG, font=dict(family=FONT),
    )
    return fig

fig04 = create_cluster_map(cl_map)
fig04.update_layout(height=520)
save(fig04, '03_cluster_map', w=1200, h=520)

Saved: /Users/leoss/Desktop/GitHub/Capstone/FINAL CODE RECAP/v2_algorithmic/pres/viz/03_cluster_map.png


### Chart 4 — Cluster Profile

In [8]:
# Grouped bar chart: normalised means per cluster across key variables
PROFILE_VARS = {
    'Total natural resources rents (% of GDP)': 'NR Rents',
    'Oil rents (% of GDP)':                      'Oil Rents',
    'GDP per capita (constant prices, PPP)':     'GDP per capita',
    'Agriculture, forestry, and fishing, value added (% of GDP)': 'Agriculture',
    'Domestic credit to private sector (% of GDP)': 'Domestic Credit',
    'Gross fixed capital formation, all, Constant prices, Percent of GDP': 'GFCF',
    'Human capital index':                       'Human Capital',
    'Life expectancy at birth, total (years)':   'Life Expectancy',
    'Rule of law index':                         'Rule of Law',
    'Political corruption index':                'Pol. Corruption',
}

# Compute cluster means, then normalise 0-1 within each variable
available = {k: v for k, v in PROFILE_VARS.items() if k in panel.columns}
cluster_means = panel.groupby('ClusterLabels')[list(available.keys())].mean()

# Min-max normalise — .where() prevents the division ever touching zero-range columns
_lo  = cluster_means.min()
_rng = cluster_means.max() - _lo          # range per variable
_safe = _rng.where(_rng > 0)              # NaN where range == 0; no division happens
normed = cluster_means.sub(_lo).div(_safe).fillna(0.5)

normed = normed.rename(columns=available)

fig06 = go.Figure()
for lbl in sorted(normed.index):
    color = CLUSTER_COLORS.get(lbl, '#999')
    fig06.add_trace(go.Bar(
        x=normed.columns.tolist(),
        y=normed.loc[lbl].values,
        name=lbl,
        marker=dict(color=color, opacity=0.88),
        hovertemplate='%{x}: %{y:.2f}<extra>' + lbl + '</extra>',
    ))

fig06.update_layout(**base_layout(
    barmode='group', height=500,
    margin=dict(l=60, r=40, t=60, b=120),
    xaxis=dict(tickangle=-30, tickfont=dict(size=10)),
    yaxis=dict(title='Normalised mean (0 = sample min, 1 = sample max)',
               gridcolor=GRID, gridwidth=0.5, range=[0, 1.05]),
    legend=dict(orientation='h', yanchor='bottom', y=1.02,
                xanchor='center', x=0.5, font=dict(size=10)),
))
save(fig06, '04_cluster_profile', w=1200, h=500)


Saved: /Users/leoss/Desktop/GitHub/Capstone/FINAL CODE RECAP/v2_algorithmic/pres/viz/04_cluster_profile.png


### Chart 5 — ECI Distribution Shift

In [9]:
yr_95 = panel[panel['Year'] == 1995]['Economic Complexity Index'].dropna().sort_values().values
yr_19 = panel[panel['Year'] == 2019]['Economic Complexity Index'].dropna().sort_values().values

fig14 = go.Figure()
for vals, yr, col in [(yr_95, 1995, '#457B9D'), (yr_19, 2019, '#E63946')]:
    pcts = np.linspace(0, 100, len(vals))
    fig14.add_trace(go.Scatter(
        x=pcts, y=vals, mode='lines', name=str(yr),
        line=dict(color=col, width=2.5),
        hovertemplate=f'{yr} | P%{{x:.0f}}: %{{y:.3f}}<extra></extra>',
    ))
fig14.update_layout(**base_layout(
    height=500,
    xaxis=dict(title='Percentile', gridcolor=GRID, gridwidth=0.5),
    yaxis=dict(title='Economic Complexity Index', gridcolor=GRID, gridwidth=0.5,
               zeroline=True, zerolinecolor='#ddd', zerolinewidth=1),
    legend=dict(font=dict(size=11)),
))
save(fig14, '05_eci_dist_shift', w=1100, h=500)


Saved: /Users/leoss/Desktop/GitHub/Capstone/FINAL CODE RECAP/v2_algorithmic/pres/viz/05_eci_dist_shift.png


### Chart 6 — ECI Trajectory by Cluster

In [10]:
traj = panel.groupby(['Year', 'ClusterLabels'])['Economic Complexity Index'].median().reset_index()

fig15 = go.Figure()
for lbl in sorted(traj['ClusterLabels'].dropna().unique()):
    sub = traj[traj['ClusterLabels'] == lbl]
    fig15.add_trace(go.Scatter(
        x=sub['Year'], y=sub['Economic Complexity Index'],
        mode='lines+markers', name=lbl,
        line=dict(color=CLUSTER_COLORS.get(lbl, '#999'), width=2.2),
        marker=dict(size=5),
        hovertemplate='%{x}: %{y:.3f}<extra>' + lbl + '</extra>',
    ))
fig15.update_layout(**base_layout(
    height=480,
    xaxis=dict(title='Year', gridcolor=GRID, gridwidth=0.5, dtick=5),
    yaxis=dict(title='Median ECI', gridcolor=GRID, gridwidth=0.5,
               zeroline=True, zerolinecolor='#ddd', zerolinewidth=1),
    legend=dict(font=dict(size=10), bgcolor='rgba(255,255,255,0.9)',
                bordercolor=GRID, borderwidth=1),
    hovermode='x unified',
))
save(fig15, '06_eci_trajectory', w=1100, h=480)


Saved: /Users/leoss/Desktop/GitHub/Capstone/FINAL CODE RECAP/v2_algorithmic/pres/viz/06_eci_trajectory.png


---
## Section 2 — ML (NB8)

In [11]:
import os, warnings
import numpy as np
import pandas as pd
from sklearn.linear_model import LassoCV, RidgeCV, ElasticNetCV
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error
import plotly.graph_objects as go
from plotly.subplots import make_subplots
warnings.filterwarnings('ignore')

# ── Project root ─────────────────────────────────────────────────────────────

# ── Output ───────────────────────────────────────────────────────────────────

# ── Style ────────────────────────────────────────────────────────────────────
FONT = 'IBM Plex Sans, -apple-system, BlinkMacSystemFont, sans-serif'
BG   = '#ffffff'
NAVY = '#1a2744'
GRID = '#e5e7eb'
CFG  = {'displayModeBar': False, 'responsive': True}

PAL = {'blue': '#4a6fa5', 'red': '#c23a3a', 'green': '#2e7d4a',
       'gold': '#c9a227', 'grey': '#999999', 'orange': '#d4853b'}

CLUSTER_COLORS = {
    'Petrostates':        '#E63946',
    'Oil Exporters':      '#457B9D',
    'Major Producers':    '#2A9D8F',
    'Mining Exporters':   '#E9C46A',
    'Forestry Intensive': '#8B5CF6',
}

def base_layout(**kw):
    d = dict(template='plotly_white', plot_bgcolor=BG, paper_bgcolor=BG,
             font=dict(family=FONT, size=11, color=NAVY),
             margin=dict(l=160, r=40, t=10, b=50))
    d.update(kw)
    return d

def save(fig, name, w=1100, h=600):
    path = os.path.join(OUT, name)
    fig.write_html(f"{path}.html", config=CFG)
    print(f"  Saved: {path}.html")
    try:
        fig.write_image(f"{path}.png", width=w, height=h, scale=3)
        print(f"  Saved: {path}.png")
    except Exception:
        print(f"  (PNG skipped — pip install kaleido)")

print(f'Root: {ROOT}')
print(f'Output: {OUT}')

Root: /Users/leoss/Desktop/GitHub/Capstone/FINAL CODE RECAP/v2_algorithmic
Output: /Users/leoss/Desktop/GitHub/Capstone/FINAL CODE RECAP/v2_algorithmic/pres/viz


In [12]:
# ── Redirect all saves to outputs/charts/ ────────────────────────────────────
import os as _os
_CHARTS = OUT

def save(fig, name, w=1100, h=600):
    path = _os.path.join(_CHARTS, name)
    # fig.write_html(f'{path}.html')  # disabled: save disk space
    try:
        fig.write_image(f'{path}.png', width=w, height=h, scale=2)
        print(f'  Saved: {path}.png')
    except Exception as e:
        print(f'  PNG skipped: {e}')


In [13]:
master = pd.read_csv(os.path.join(DATA_ROOT, 'intermediary/Master.csv'))
include_list = pd.read_csv(os.path.join(DATA_ROOT, 'intermediary/high_resource_countries.csv'))['Country Code'].unique().tolist()
df = master[(master['Year'].between(1995, 2019)) &
            (master['Country Code'].isin(include_list))].copy()
df = df.sort_values(['Country Code', 'Year']).reset_index(drop=True)

clusters = pd.read_csv(os.path.join(DATA_ROOT, 'intermediary/clusters_k5_agg.csv'))
df = df.merge(clusters[['Country Code', 'ClusterLabels']].drop_duplicates('Country Code'),
              on='Country Code', how='left')

df['Total_Production_Value_Per_Capita'] = (
    df['Total_Production_Value'] / df['Population'].replace(0, np.nan))
df['L1_ECI']    = df.groupby('Country Code')['Economic Complexity Index'].shift(1)
df['ECI_delta'] = df['Economic Complexity Index'] - df['L1_ECI']
df = df.dropna(subset=['L1_ECI', 'Economic Complexity Index', 'ECI_delta'])

log_cols = [
    'Human capital index',
    'Total_Production_Value_Per_Capita',
    'Gross fixed capital formation, all, Constant prices, Percent of GDP',
    'Government revenue',
    'Use of IMF credit (DOD, current US$)',
]
df[log_cols] = np.log1p(df[log_cols]).replace([np.inf, -np.inf], np.nan)

hci_m  = df['Human capital index'].mean()
prod_m = df['Total_Production_Value_Per_Capita'].mean()
gfcf_m = df['Gross fixed capital formation, all, Constant prices, Percent of GDP'].mean()
df['HCI_x_ProductionValue']  = (df['Human capital index'] - hci_m) * \
                                (df['Total_Production_Value_Per_Capita'] - prod_m)
df['GFCF_x_ProductionValue'] = (df['Gross fixed capital formation, all, Constant prices, Percent of GDP'] - gfcf_m) * \
                                (df['Total_Production_Value_Per_Capita'] - prod_m)

base_features = [
    'Total_Production_Value_Per_Capita', 'Human capital index',
    'Rule of law index', 'Political stability \u2014 estimate',
    'Trade (% of GDP)',
    'Gross fixed capital formation, all, Constant prices, Percent of GDP',
    'Share of investment in GDP', 'Domestic credit to private sector (% of GDP)',
    'Landlocked', 'Urban population (% of total population)',
    'Government revenue', 'Capital depreciation rate',
    'Use of IMF credit (DOD, current US$)', 'Real interest rate (%)',
    'Inflation, consumer prices (annual %)', 'Access to electricity (% of population)',
    'Adjusted savings: gross savings (% of GNI)', 'L1_ECI',
    'Forestry rents (% of GDP)',
]
all_features = base_features + ['HCI_x_ProductionValue', 'GFCF_x_ProductionValue']
df = df.dropna(subset=all_features)

name_map = {
    'Total_Production_Value_Per_Capita': 'Production Value',
    'Human capital index': 'Human Capital',
    'Rule of law index': 'Rule of Law',
    'Political stability \u2014 estimate': 'Political Stability',
    'Trade (% of GDP)': 'Trade',
    'Gross fixed capital formation, all, Constant prices, Percent of GDP': 'Capital Formation',
    'Share of investment in GDP': 'Investment Share',
    'Domestic credit to private sector (% of GDP)': 'Domestic Credit',
    'Landlocked': 'Landlocked',
    'Urban population (% of total population)': 'Urban Population',
    'Government revenue': 'Gov Revenue',
    'Capital depreciation rate': 'Depreciation',
    'Use of IMF credit (DOD, current US$)': 'IMF Credit',
    'Real interest rate (%)': 'Real Rate',
    'Inflation, consumer prices (annual %)': 'Inflation',
    'Access to electricity (% of population)': 'Electricity',
    'Adjusted savings: gross savings (% of GNI)': 'Gross Savings',
    'L1_ECI': 'Lagged ECI',
    'HCI_x_ProductionValue': 'HC \u00d7 Production',
    'GFCF_x_ProductionValue': 'GFCF \u00d7 Production',
}
short = [name_map.get(f, f) for f in all_features]
EXCLUDE_LABEL = 'Lagged ECI'  # trivially dominant, excluded from importance charts

print(f'Sample: {df["Country Code"].nunique()} countries, {len(df):,} obs, {len(all_features)} features')

Sample: 73 countries, 1,716 obs, 21 features


In [14]:
class PanelTemporalCV:
    def __init__(self, years, n_splits=5, gap=1, min_train_years=8):
        self.years = np.asarray(years)
        uy = np.sort(np.unique(self.years))
        ec = uy[0] + min_train_years - 1
        lc = uy[-1] - gap - 1
        self.cutoffs = np.unique(np.linspace(ec, lc, n_splits).astype(int))
        self.n_splits = len(self.cutoffs)
        self.gap = gap
    def split(self, X=None, y=None, groups=None):
        for c in self.cutoffs:
            ti = np.where(self.years <= c)[0]
            vi = np.where(self.years > c + self.gap)[0]
            if len(ti) and len(vi): yield ti, vi
    def get_n_splits(self, X=None, y=None, groups=None): return self.n_splits

train_df = df[df['Year'] <= 2014].copy()
test_df  = df[df['Year'] >= 2015].copy()

y_tr_lv = train_df['Economic Complexity Index'].values
y_te_lv = test_df['Economic Complexity Index'].values
y_tr_dl = train_df['ECI_delta'].values
y_te_dl = test_df['ECI_delta'].values

scaler  = StandardScaler()
X_train = scaler.fit_transform(train_df[all_features].values)
X_test  = scaler.transform(test_df[all_features].values)

tscv = PanelTemporalCV(train_df['Year'].values, n_splits=5, gap=1, min_train_years=8)

# ── ECI level models ─────────────────────────────────────────────────────────
print('Fitting models (ECI level)...')
lasso   = LassoCV(cv=tscv, random_state=42, max_iter=10000).fit(X_train, y_tr_lv)
ridge   = RidgeCV(alphas=np.logspace(-3, 3, 100), cv=tscv).fit(X_train, y_tr_lv)
elastic = ElasticNetCV(l1_ratio=[0.5], cv=tscv, random_state=42, max_iter=10000).fit(X_train, y_tr_lv)
rf      = RandomForestRegressor(n_estimators=200, max_depth=4, min_samples_leaf=10,
                                 random_state=42, n_jobs=-1, oob_score=True).fit(X_train, y_tr_lv)
models_lv = {'LASSO': lasso, 'Ridge': ridge, 'Elastic Net': elastic, 'Random Forest': rf}
print(f'  LASSO \u03b1={lasso.alpha_:.4f}  Ridge \u03b1={ridge.alpha_:.4f}  '
      f'EN \u03b1={elastic.alpha_:.4f}  RF OOB={rf.oob_score_:.3f}')

# ── ΔECI models ──────────────────────────────────────────────────────────────
print('Fitting models (\u0394ECI)...')
lasso_d   = LassoCV(cv=tscv, random_state=42, max_iter=10000).fit(X_train, y_tr_dl)
ridge_d   = RidgeCV(alphas=np.logspace(-3, 3, 100), cv=tscv).fit(X_train, y_tr_dl)
elastic_d = ElasticNetCV(l1_ratio=[0.5], cv=tscv, random_state=42, max_iter=10000).fit(X_train, y_tr_dl)
rf_d      = RandomForestRegressor(n_estimators=200, max_depth=4, min_samples_leaf=10,
                                   random_state=42, n_jobs=-1, oob_score=True).fit(X_train, y_tr_dl)
models_dl = {'LASSO': lasso_d, 'Ridge': ridge_d, 'Elastic Net': elastic_d, 'Random Forest': rf_d}

# ── Performance tables ───────────────────────────────────────────────────────
def eval_models(models, X_tr, y_tr, X_te, y_te):
    rows = []
    for name, m in models.items():
        tr_r2 = r2_score(y_tr, m.predict(X_tr))
        te_r2 = r2_score(y_te, m.predict(X_te))
        rows.append({'Model': name, 'Train R\u00b2': round(tr_r2, 4),
                     'Test R\u00b2': round(te_r2, 4),
                     'Overfit Gap': round(tr_r2 - te_r2, 4)})
    return pd.DataFrame(rows).sort_values('Test R\u00b2', ascending=False).reset_index(drop=True)

perf_lv = eval_models(models_lv, X_train, y_tr_lv, X_test, y_te_lv)
perf_dl = eval_models(models_dl, X_train, y_tr_dl, X_test, y_te_dl)

print('\n-- ECI Level --')
print(perf_lv.to_string(index=False))
print('\n-- \u0394ECI --')
print(perf_dl.to_string(index=False))

# ── Normalised importance ────────────────────────────────────────────────────
def minmax(a):
    lo, hi = a.min(), a.max()
    return (a - lo) / (hi - lo + 1e-12)

imp = pd.DataFrame({'Feature': all_features, 'Short': short})
imp['LASSO']       = minmax(np.abs(lasso.coef_))
imp['Ridge']       = minmax(np.abs(ridge.coef_))
imp['Elastic Net'] = minmax(np.abs(elastic.coef_))
imp['RF']          = minmax(rf.feature_importances_)
imp['avg']         = imp[['LASSO', 'Ridge', 'Elastic Net', 'RF']].mean(axis=1)
imp_show = imp[imp['Short'] != EXCLUDE_LABEL].sort_values('avg', ascending=False).reset_index(drop=True)

print(f'\nModels fitted. Train: {len(train_df):,} obs | Test: {len(test_df):,} obs')

Fitting models (ECI level)...


  LASSO α=0.0077  Ridge α=0.0010  EN α=0.0143  RF OOB=0.843
Fitting models (ΔECI)...



-- ECI Level --
        Model  Train R²  Test R²  Overfit Gap
Random Forest    0.8705   0.9000      -0.0295
        LASSO    0.8481   0.8973      -0.0492
  Elastic Net    0.8483   0.8963      -0.0479
        Ridge    0.8501   0.8926      -0.0425

-- ΔECI --
        Model  Train R²  Test R²  Overfit Gap
Random Forest    0.2277   0.1357       0.0920
  Elastic Net    0.0984   0.0625       0.0359
        LASSO    0.0994   0.0614       0.0381
        Ridge    0.1074   0.0492       0.0582

Models fitted. Train: 1,351 obs | Test: 365 obs


### Chart 7 — ML Feature Importance

In [15]:
train_df = df[df['Year'] <= 2014].copy()
test_df  = df[df['Year'] >= 2015].copy()

y_tr_lv = train_df['Economic Complexity Index'].values
y_te_lv = test_df['Economic Complexity Index'].values
y_tr_dl = train_df['ECI_delta'].values
y_te_dl = test_df['ECI_delta'].values

scaler  = StandardScaler()
X_train = scaler.fit_transform(train_df[all_features].values)
X_test  = scaler.transform(test_df[all_features].values)

tscv = PanelTemporalCV(train_df['Year'].values, n_splits=5, gap=1, min_train_years=8)

print('Fitting models (ECI level)...')
lasso   = LassoCV(cv=tscv, random_state=42, max_iter=10000).fit(X_train, y_tr_lv)
ridge   = RidgeCV(alphas=np.logspace(-3, 3, 100), cv=tscv).fit(X_train, y_tr_lv)
elastic = ElasticNetCV(l1_ratio=[0.5], cv=tscv, random_state=42, max_iter=10000).fit(X_train, y_tr_lv)
rf      = RandomForestRegressor(n_estimators=200, max_depth=4, min_samples_leaf=10,
                                 random_state=42, n_jobs=-1, oob_score=True).fit(X_train, y_tr_lv)
models_lv = {'LASSO': lasso, 'Ridge': ridge, 'Elastic Net': elastic, 'Random Forest': rf}
print(f"  LASSO alpha={lasso.alpha_:.4f}  Ridge alpha={ridge.alpha_:.4f}  "
      f"EN alpha={elastic.alpha_:.4f}  RF OOB={rf.oob_score_:.3f}")

print('Fitting models (delta ECI)...')
lasso_d   = LassoCV(cv=tscv, random_state=42, max_iter=10000).fit(X_train, y_tr_dl)
ridge_d   = RidgeCV(alphas=np.logspace(-3, 3, 100), cv=tscv).fit(X_train, y_tr_dl)
elastic_d = ElasticNetCV(l1_ratio=[0.5], cv=tscv, random_state=42, max_iter=10000).fit(X_train, y_tr_dl)
rf_d      = RandomForestRegressor(n_estimators=200, max_depth=4, min_samples_leaf=10,
                                   random_state=42, n_jobs=-1, oob_score=True).fit(X_train, y_tr_dl)
models_dl = {'LASSO': lasso_d, 'Ridge': ridge_d, 'Elastic Net': elastic_d, 'Random Forest': rf_d}

def eval_models(models, X_tr, y_tr, X_te, y_te):
    rows = []
    for name, m in models.items():
        tr_r2 = r2_score(y_tr, m.predict(X_tr))
        te_r2 = r2_score(y_te, m.predict(X_te))
        rows.append({'Model': name, 'Train R²': round(tr_r2, 4),
                     'Test R²': round(te_r2, 4),
                     'Overfit Gap': round(tr_r2 - te_r2, 4)})
    return pd.DataFrame(rows).sort_values('Test R²', ascending=False).reset_index(drop=True)

perf_lv = eval_models(models_lv, X_train, y_tr_lv, X_test, y_te_lv)
perf_dl = eval_models(models_dl, X_train, y_tr_dl, X_test, y_te_dl)
print('-- ECI Level --'); print(perf_lv.to_string(index=False))
print('-- dECI --');      print(perf_dl.to_string(index=False))

def minmax(a):
    lo, hi = a.min(), a.max()
    return (a - lo) / (hi - lo + 1e-12)

imp = pd.DataFrame({'Feature': all_features, 'Short': short})
imp['LASSO']       = minmax(np.abs(lasso.coef_))
imp['Ridge']       = minmax(np.abs(ridge.coef_))
imp['Elastic Net'] = minmax(np.abs(elastic.coef_))
imp['RF']          = minmax(rf.feature_importances_)
imp['avg']         = imp[['LASSO', 'Ridge', 'Elastic Net', 'RF']].mean(axis=1)
imp_show = imp[imp['Short'] != EXCLUDE_LABEL].sort_values('avg', ascending=False).reset_index(drop=True)

print(f"Models fitted. Train: {len(train_df):,} | Test: {len(test_df):,} obs")


Fitting models (ECI level)...


  LASSO alpha=0.0077  Ridge alpha=0.0010  EN alpha=0.0143  RF OOB=0.843
Fitting models (delta ECI)...


-- ECI Level --
        Model  Train R²  Test R²  Overfit Gap
Random Forest    0.8705   0.9000      -0.0295
        LASSO    0.8481   0.8973      -0.0492
  Elastic Net    0.8483   0.8963      -0.0479
        Ridge    0.8501   0.8926      -0.0425
-- dECI --
        Model  Train R²  Test R²  Overfit Gap
Random Forest    0.2277   0.1357       0.0920
  Elastic Net    0.0984   0.0625       0.0359
        LASSO    0.0994   0.0614       0.0381
        Ridge    0.1074   0.0492       0.0582
Models fitted. Train: 1,351 | Test: 365 obs


In [16]:
d = imp_show.head(15).iloc[::-1].reset_index(drop=True)
lin_models = ['LASSO', 'Ridge', 'Elastic Net']
symbols = {'LASSO': 'circle', 'Ridge': 'square', 'Elastic Net': 'triangle-up'}
colors  = {'LASSO': PAL['red'], 'Ridge': PAL['blue'], 'Elastic Net': PAL['green']}

fig7 = go.Figure()
for _, row in d.iterrows():
    vals = [row[m] for m in lin_models if not np.isnan(row[m])]
    if len(vals) >= 2:
        fig7.add_shape(type='line',
                       x0=min(vals), x1=max(vals), y0=row['Short'], y1=row['Short'],
                       line=dict(color='#b0c0d4', width=2))

for m in lin_models:
    fig7.add_trace(go.Scatter(
        x=d[m], y=d['Short'], mode='markers', name=m,
        marker=dict(symbol=symbols[m], color=colors[m], size=11,
                    line=dict(width=1.2, color='white')),
        hovertemplate=f'%{{y}}: %{{x:.3f}}<extra>{m}</extra>',
    ))

fig7.update_layout(**base_layout(
    height=560, margin=dict(l=180, r=60, t=40, b=60),
    xaxis=dict(title='Normalised Feature Importance (min-max, 0\u20131)',
               gridcolor=GRID, gridwidth=0.5, range=[-0.02, 0.27]),
    yaxis=dict(tickfont=dict(size=11)),
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='center', x=0.5,
                font=dict(size=11)),
))
save(fig7, '07_ml_feature_importance', w=1200, h=560)
fig7.show(config=CFG)

  Saved: /Users/leoss/Desktop/GitHub/Capstone/FINAL CODE RECAP/v2_algorithmic/pres/viz/07_ml_feature_importance.png


### Chart 8 — ML Train vs Test R²

In [17]:
fig9 = make_subplots(rows=1, cols=2, horizontal_spacing=0.12,
                     subplot_titles=['ECI Level', '\u0394ECI'])

for col_idx, perf in enumerate([perf_lv, perf_dl], 1):
    show = (col_idx == 1)
    fig9.add_trace(go.Bar(
        x=perf['Model'], y=perf['Train R\u00b2'], name='Train R\u00b2',
        marker_color=PAL['blue'], opacity=0.85, legendgroup='train',
        text=[f'{v:.3f}' for v in perf['Train R\u00b2']], textposition='outside',
        textfont=dict(size=10, color=PAL['blue']),
        showlegend=show,
    ), row=1, col=col_idx)
    fig9.add_trace(go.Bar(
        x=perf['Model'], y=perf['Test R\u00b2'], name='Test R\u00b2',
        marker_color=PAL['red'], opacity=0.85, legendgroup='test',
        text=[f'{v:.3f}' for v in perf['Test R\u00b2']], textposition='outside',
        textfont=dict(size=10, color=PAL['red']),
        showlegend=show,
    ), row=1, col=col_idx)
    fig9.update_xaxes(tickangle=-25, tickfont=dict(size=10), row=1, col=col_idx)
    fig9.update_yaxes(title_text='R\u00b2' if col_idx == 1 else '',
                      gridcolor=GRID, gridwidth=0.5, row=1, col=col_idx)

fig9.update_layout(
    template='plotly_white', plot_bgcolor=BG, paper_bgcolor=BG,
    font=dict(family=FONT, size=11, color=NAVY),
    barmode='group', height=480, margin=dict(l=60, r=40, t=50, b=80),
    legend=dict(orientation='h', yanchor='bottom', y=1.06, xanchor='center', x=0.5,
                font=dict(size=12)),
)
save(fig9, '08_ml_r2', w=1200, h=480)
fig9.show(config=CFG)

  Saved: /Users/leoss/Desktop/GitHub/Capstone/FINAL CODE RECAP/v2_algorithmic/pres/viz/08_ml_r2.png


---
## Section 3 — Regression (NB9)

In [18]:
import os, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from sklearn.metrics import r2_score
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ── Project root ──────────────────────────────────────────────────────────────


# ── Style ─────────────────────────────────────────────────────────────────────
FONT = 'IBM Plex Sans, -apple-system, BlinkMacSystemFont, sans-serif'
BG   = '#ffffff'
NAVY = '#1a2744'
GRID = '#e5e7eb'
CFG  = {'displayModeBar': False, 'responsive': True}

SPEC_COLORS = {
    '3a': '#4a6fa5', '3b': '#c23a3a', '3c': '#2e7d4a',
    '3d': '#d4853b', '3e': '#7b6fa5',
}
CLUSTER_COLORS = {
    'Petrostates':           '#E63946',
    'Oil Exporters':         '#457B9D',
    'Diversified Exporters': '#2A9D8F',
    'Gold & Coal':           '#E9C46A',
}

def base_layout(**kw):
    d = dict(template='plotly_white', plot_bgcolor=BG, paper_bgcolor=BG,
             font=dict(family=FONT, size=11, color=NAVY),
             margin=dict(l=60, r=40, t=50, b=60))
    d.update(kw)
    return d

def save(fig, name, w=1100, h=600):
    path = os.path.join(OUT, name)
    fig.write_html(f"{path}.html", config=CFG)
    print(f"  Saved: {path}.html")
    try:
        fig.write_image(f"{path}.png", width=w, height=h, scale=3)
        print(f"  Saved: {path}.png")
    except Exception:
        print(f"  (PNG skipped — pip install kaleido)")

print(f'Root: {ROOT}')
print(f'Output: {OUT}')

Root: /Users/leoss/Desktop/GitHub/Capstone/FINAL CODE RECAP/v2_algorithmic
Output: /Users/leoss/Desktop/GitHub/Capstone/FINAL CODE RECAP/v2_algorithmic/pres/viz


In [19]:
# ── Redirect all saves to Final/charts/regression/ ────────────────────────────────────
import os as _os
_CHARTS = OUT

def save(fig, name, w=1100, h=600):
    path = _os.path.join(_CHARTS, name)
    # fig.write_html(f'{path}.html')  # disabled: save disk space
    try:
        fig.write_image(f'{path}.png', width=w, height=h, scale=2)
        print(f'  Saved: {path}.png')
    except Exception as e:
        print(f'  PNG skipped: {e}')


In [20]:
ECI_COL = 'Economic Complexity Index'

master   = pd.read_csv(os.path.join(DATA_ROOT, 'intermediary/Master.csv'), dtype={'Country Code': str})
clusters = pd.read_csv(os.path.join(DATA_ROOT, 'intermediary/clustersagg.csv'), dtype={'Country Code': str})

cl_map = clusters[['Country Code', 'Cluster', 'ClusterLabels']].drop_duplicates('Country Code')
df = master.merge(cl_map, on='Country Code', how='inner')
df['Year'] = df['Year'].astype(int)
df = df.sort_values(['Country Code', 'Year']).reset_index(drop=True)

# ── Feature engineering ───────────────────────────────────────────────────────
df['Total_Production_Value_Per_Capita'] = (
    df['Total_Production_Value'] / df['Population'].replace(0, np.nan))

df['log_HCI']              = np.log1p(df['Human capital index'].clip(lower=0))
df['log_GFCF']             = np.log1p(
    df['Gross fixed capital formation, all, Constant prices, Percent of GDP'].clip(lower=0))
df['log_Production_Value'] = np.log1p(df['Total_Production_Value_Per_Capita'].clip(lower=0))

df['ECI_lag1']  = df.groupby('Country Code')[ECI_COL].shift(1)
df['delta_ECI'] = df[ECI_COL] - df['ECI_lag1']

BASE_INDEP = [
    'log_HCI', 'log_GFCF',
    'Political stability \u2014 estimate',
    'Rule of law index',
    'log_Production_Value',
    'Trade (% of GDP)',
]
EXTRA_CONTROLS = [
    'Hydrocarbons_Dominant', 'Subsoil_Metals_Dominant',
    'Precious_Metals_Dominant', 'Access to electricity (% of population)',
]

for var in BASE_INDEP:
    df[f'{var}_lag1'] = df.groupby('Country Code')[var].shift(1)

hci_c  = df['log_HCI']             - df['log_HCI'].mean()
gfcf_c = df['log_GFCF']            - df['log_GFCF'].mean()
prod_c = df['log_Production_Value'] - df['log_Production_Value'].mean()
df['log_HCI_x_log_Production']  = hci_c  * prod_c
df['log_GFCF_x_log_Production'] = gfcf_c * prod_c

hci_l_c  = df['log_HCI_lag1']             - df['log_HCI_lag1'].mean()
gfcf_l_c = df['log_GFCF_lag1']            - df['log_GFCF_lag1'].mean()
prod_l_c = df['log_Production_Value_lag1'] - df['log_Production_Value_lag1'].mean()
df['log_HCI_x_log_Production_lag1']  = hci_l_c  * prod_l_c
df['log_GFCF_x_log_Production_lag1'] = gfcf_l_c * prod_l_c

DISPLAY_LABELS = {
    'const':                                           'Constant',
    'log_HCI':                                         'Human Capital (log)',
    'log_GFCF':                                        'GFCF (log)',
    'Political stability \u2014 estimate':            'Political Stability',
    'Rule of law index':                               'Rule of Law',
    'log_Production_Value':                            'NR Production (log, pc)',
    'Trade (% of GDP)':                                'Trade (% GDP)',
    'log_HCI_x_log_Production':                        'HCI \u00d7 Production',
    'log_GFCF_x_log_Production':                       'GFCF \u00d7 Production',
    'ECI_lag1':                                        'ECI (t-1)',
    'log_HCI_lag1':                                    'Human Capital (t-1)',
    'log_GFCF_lag1':                                   'GFCF (t-1)',
    'Political stability \u2014 estimate_lag1':       'Political Stability (t-1)',
    'Rule of law index_lag1':                          'Rule of Law (t-1)',
    'log_Production_Value_lag1':                       'NR Production (t-1)',
    'Trade (% of GDP)_lag1':                           'Trade (t-1)',
    'log_HCI_x_log_Production_lag1':                   'HCI \u00d7 Production (t-1)',
    'log_GFCF_x_log_Production_lag1':                  'GFCF \u00d7 Production (t-1)',
    'Hydrocarbons_Dominant':                           'Hydrocarbons dominant',
    'Subsoil_Metals_Dominant':                         'Subsoil metals dominant',
    'Precious_Metals_Dominant':                        'Precious metals dominant',
    'Access to electricity (% of population)':        'Electricity access',
}

print(f'Sample: {df["Country Code"].nunique()} countries, {len(df):,} obs')
print(f'Years: {df["Year"].min()}\u2013{df["Year"].max()}')
print(f'Cluster distribution:')
print(df.drop_duplicates("Country Code")["ClusterLabels"].value_counts().to_string())


Sample: 74 countries, 1,850 obs
Years: 1995–2019
Cluster distribution:
ClusterLabels
Forestry Intensive    37
Oil Exporters         16
Major Producers       10
Petrostates            7
Mining Exporters       4


In [21]:
# ── Load pre-computed regression results from NB6 ────────────────────────────
NB6_OUT = '/Users/leoss/Desktop/GitHub/Capstone/FINAL CODE RECAP/v2_algorithmic/robustness-forest/outputs/regression'
MODEL_ORDER  = ['3a', '3b', '3c', '3d', '3e']
MODEL_LABELS = {
    '3a': '3a Base', '3b': '3b + Lag',
    '3c': '3c All Lagged', '3d': '3d Extended', '3e': '3e ΔECI',
}

coef_dfs = {}
model_stats = {}
for m in MODEL_ORDER:
    p = f'{NB6_OUT}/coef_model{m}.csv'
    if os.path.exists(p):
        _tmp = pd.read_csv(p)
        coef_dfs[m] = _tmp
        r2_val = float(_tmp['R2'].iloc[0]) if 'R2' in _tmp.columns else None
        n_val  = int(_tmp['N'].iloc[0])    if 'N'  in _tmp.columns else None
        model_stats[m] = {'R2': r2_val, 'N': n_val, 'label': MODEL_LABELS[m]}
        print(f'  Loaded model {m}: {len(_tmp)} rows  R²={r2_val}  N={n_val}')
    else:
        print(f'  WARNING: missing {p}')
        model_stats[m] = {'R2': None, 'N': None, 'label': MODEL_LABELS[m]}

  Loaded model 3a: 9 rows  R²=0.297  N=1810
  Loaded model 3b: 10 rows  R²=0.8517  N=1740
  Loaded model 3c: 10 rows  R²=0.8539  N=1736
  Loaded model 3d: 12 rows  R²=0.8549  N=1716
  Loaded model 3e: 12 rows  R²=0.077  N=1716


### Chart 9 — Coefficient Forest

In [22]:
FOREST_VARS = [
    'log_HCI', 'log_GFCF',
    'Political stability \u2014 estimate', 'Rule of law index',
    'log_Production_Value', 'Trade (% of GDP)',
    'log_HCI_x_log_Production', 'log_GFCF_x_log_Production',
    'ECI_lag1',
    'Hydrocarbons_Dominant', 'Subsoil_Metals_Dominant',
    'Precious_Metals_Dominant', 'Access to electricity (% of population)',
]


# Drop variables absent from all loaded models (absorbed / not in any spec)
_all_vars = set()
for _df in coef_dfs.values():
    _all_vars.update(_df['Variable'].tolist())
FOREST_VARS = [v for v in FOREST_VARS if v in _all_vars]

y_labels = [DISPLAY_LABELS.get(v, v) for v in FOREST_VARS]
y_pos    = {lbl: i for i, lbl in enumerate(y_labels)}
n_specs  = len(MODEL_ORDER)
offsets  = np.linspace(-0.22, 0.22, n_specs)

fig16 = go.Figure()
fig16.add_vline(x=0, line=dict(color='#aab0ba', width=1.5, dash='dash'))

# Horizontal reference bands
for i in range(len(y_labels)):
    if i % 2 == 0:
        fig16.add_hrect(y0=i - 0.48, y1=i + 0.48,
                        fillcolor='rgba(230,235,242,0.35)', line=dict(width=0), layer='below')

for i, m in enumerate(MODEL_ORDER):
    if m not in coef_dfs: continue
    tbl = coef_dfs[m]
    col = SPEC_COLORS[m]
    lbl = MODEL_LABELS[m]

    matched = tbl[tbl['Variable'].isin(FOREST_VARS)].copy()
    matched['Label'] = matched['Variable'].map(lambda v: DISPLAY_LABELS.get(v, v))
    matched['y_pos'] = matched['Label'].map(y_pos).fillna(-1) + offsets[i]
    matched = matched[matched['y_pos'] >= 0]

    fig16.add_trace(go.Scatter(
        x=matched['Coef'], y=matched['y_pos'],
        mode='markers',
        marker=dict(size=9, color=col, line=dict(color='white', width=1.5)),
        error_x=dict(
            type='data', symmetric=False,
            array=(matched['CI_hi'] - matched['Coef']).values,
            arrayminus=(matched['Coef'] - matched['CI_lo']).values,
            color=col, thickness=2.0, width=5,
        ),
        name=lbl,
        text=matched['Label'],
        hovertemplate='<b>%{text}</b><br>\u03b2 = %{x:.4f}<extra>' + lbl + '</extra>',
    ))

fig16.update_layout(**base_layout(
    height=max(560, len(y_labels) * 50 + 150),
    margin=dict(l=220, r=60, t=55, b=60),
    xaxis=dict(title='Coefficient (95% CI, SE clustered by country)',
               gridcolor=GRID, gridwidth=0.5, zeroline=False),
    yaxis=dict(
        tickvals=list(y_pos.values()),
        ticktext=list(y_pos.keys()),
        tickfont=dict(size=11), showgrid=True, gridcolor=GRID, gridwidth=0.5,
        range=[-0.55, len(y_labels) - 0.45],
    ),
    legend=dict(orientation='h', yanchor='bottom', y=1.02,
                xanchor='center', x=0.5, font=dict(size=11)),
))
save(fig16, '09_reg_coef_forest', w=1200, h=max(560, len(y_labels) * 50 + 150))
fig16.show(config=CFG)


  Saved: /Users/leoss/Desktop/GitHub/Capstone/FINAL CODE RECAP/v2_algorithmic/pres/viz/09_reg_coef_forest.png


### Chart 10 — R² Comparison

In [23]:
perf_rows = []
for m in MODEL_ORDER:
    if m not in model_stats or model_stats[m]['R2'] is None: continue
    perf_rows.append({
        'Model': MODEL_LABELS[m],
        'R²': model_stats[m]['R2'],
        'N': model_stats[m]['N'],
        'color': SPEC_COLORS[m],
    })
perf_df = pd.DataFrame(perf_rows)

fig17 = go.Figure()
fig17.add_trace(go.Bar(
    x=perf_df['Model'], y=perf_df['R²'],
    marker=dict(color=[SPEC_COLORS[m] for m in MODEL_ORDER if m in coef_dfs],
                opacity=0.88, line=dict(color='white', width=1.5)),
    text=[f'{v:.3f}' for v in perf_df['R²']], textposition='outside',
    textfont=dict(size=11),
    hovertemplate='<b>%{x}</b><br>R² = %{y:.4f}<extra></extra>',
    name='R²',
))

# Add N annotations below bars
for _, row in perf_df.iterrows():
    fig17.add_annotation(
        x=row['Model'], y=-0.06, text=f"N={row['N']:,}", showarrow=False,
        font=dict(size=9, color='#666'), yref='y',
    )

fig17.update_layout(**base_layout(
    height=480, barmode='group',
    margin=dict(l=60, r=40, t=55, b=80),
    xaxis=dict(tickfont=dict(size=12)),
    yaxis=dict(title='R²', range=[0, 1.12], gridcolor=GRID, gridwidth=0.5),
    legend=dict(orientation='h', yanchor='bottom', y=1.02,
                xanchor='center', x=0.5, font=dict(size=11)),
))
save(fig17, '10_reg_r2', w=900, h=480)
fig17.show(config=CFG)

  Saved: /Users/leoss/Desktop/GitHub/Capstone/FINAL CODE RECAP/v2_algorithmic/pres/viz/10_reg_r2.png
